# S12 — Local and Open Models

In this session, I am learning how local and open models work and how they compare with hosted models.

I will explore Ollama, open model weights, quantization, tool calling with smaller models, and the differences between running models locally and using hosted models.

## 1. Setup

In [1]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from local_models_utils import model_size_gb, fits_on, laptop_options, load_local_catalog, probe_tool_calling, speed_table
from providers_utils import groq_client, ollama_client, ask
from landscape_utils import time_stream, PROMPT

groq = groq_client()
GROQ_MODEL = os.getenv("MODEL")
try:
    ollama = ollama_client()
    local_models = [m.id for m in ollama.models.list().data]
    print("Ollama models available:", local_models)
except Exception as e:
    ollama, local_models = None, []
    print("Ollama not reachable:", e)

Ollama models available: ['qwen2.5:7b', 'llama3.2:3b', 'nomic-embed-text:latest']


## 2. Where open models come from: Hugging Face
Ollama does not train models — it packages and runs models published on hubs like Hugging Face, the same place Groq's hosted open-weight models originate from. The names differ slightly: a model on Hugging Face as `meta-llama/Llama-3.2-3B-Instruct` becomes `llama3.2:3b` as an Ollama tag, and `openai/gpt-oss-20b` on Groq is the same underlying weights as `gpt-oss:20b` if you pulled it locally.

*Question 1 — If a Hugging Face model page says "gated: request access", what does that tell you about its licence?*

I learned that a “gated: request access” status means that the model weights are not completely open or unrestricted. Users may need to agree to specific license terms or usage conditions before getting access to the model.

In [2]:
print(load_local_catalog())

[{'id': 'llama32-1b', 'name': 'llama3.2:1b', 'params_billion': 1.0, 'precision': 'int4', 'tool_calling': False, 'notes': 'Smallest useful chat model. Fast on anything, weak reasoning.'}, {'id': 'llama32-3b', 'name': 'llama3.2:3b', 'params_billion': 3.0, 'precision': 'int4', 'tool_calling': False, 'notes': 'The course default local model. No reliable tool calling.'}, {'id': 'qwen25-7b', 'name': 'qwen2.5:7b', 'params_billion': 7.0, 'precision': 'int4', 'tool_calling': True, 'notes': 'Supports tool calling reasonably well for its size.'}, {'id': 'llama31-8b', 'name': 'llama3.1:8b', 'params_billion': 8.0, 'precision': 'int4', 'tool_calling': True, 'notes': 'Good general-purpose local model if RAM allows.'}, {'id': 'gpt-oss-20b-local', 'name': 'gpt-oss:20b (local)', 'params_billion': 20.0, 'precision': 'int4', 'tool_calling': True, 'notes': 'Same weights as the hosted Groq default. Needs a capable laptop or a small GPU.'}, {'id': 'gpt-oss-120b-local', 'name': 'gpt-oss:120b (local)', 'params

## 3. Quantization: what it actually costs in memory
Every row below is the same family of maths you saw in Session 10: parameters x bytes-per-parameter. The only new idea is precision — how many bytes each parameter uses.

In [3]:
print(f"{'precision':10} {'bytes/param':>12} {'a 20B model':>14}")
for prec in ("fp32", "fp16", "int8", "int4"):
    print(f"{prec:10} {'':>12} {model_size_gb(20, prec):>12.1f} GB")

precision   bytes/param    a 20B model
fp32                            80.0 GB
fp16                            40.0 GB
int8                            20.0 GB
int4                            10.0 GB


In [4]:
# Which catalog models fit YOUR laptop? Edit this to your real RAM.
MY_RAM_GB = 16
fits = laptop_options(MY_RAM_GB)
print(f"On a {MY_RAM_GB} GB laptop, these fit (with 2 GB headroom for everything else):")
for m in fits:
    print(f"  {m['name']:22} {model_size_gb(m['params_billion'], m['precision']):5.1f} GB   tool calling: {m['tool_calling']}")

On a 16 GB laptop, these fit (with 2 GB headroom for everything else):
  llama3.2:1b              0.5 GB   tool calling: False
  llama3.2:3b              1.5 GB   tool calling: False
  qwen2.5:7b               3.5 GB   tool calling: True
  llama3.1:8b              4.0 GB   tool calling: True
  gpt-oss:20b (local)     10.0 GB   tool calling: True


*Question 2 — Why does the table use int4 sizes for the catalog instead of fp16, when fp16 is what the model was trained at?*

I understood that FP16 models can be too large for typical consumer hardware, while 4-bit quantization reduces the model size significantly. This makes multi-billion parameter models easier to run on laptops with limited memory.

## 4. Does a small local model support tool calling?
Not every small model handles function calling reliably — some will just talk about calling the tool instead of actually emitting a structured call. This matters directly for Module 6: an agent that can't reliably call tools can't be an agent.

In [5]:
if ollama and local_models:
    for name in local_models[:3]:
        r = probe_tool_calling(ollama, name)
        print(f"{name:20} supports_tool_calling={r['supports_tool_calling']}")
        if not r["supports_tool_calling"]:
            print("   said instead:", (r.get('raw_text') or '')[:150])
else:
    print("skipped (Ollama not running)")

qwen2.5:7b           supports_tool_calling=True
llama3.2:3b          supports_tool_calling=True
nomic-embed-text:latest supports_tool_calling=False
   said instead: 


*Question 3 — If your local model fails the tool-calling probe, what are your two options for Module 6's agent labs?*

 I learned that small local models may not always handle tool calling reliably. In such cases, we can use a hosted model with better tool-calling support or choose an open-weight model that has been trained for tool calling. Structured output and validation can also improve reliability.


## 5. Local vs hosted: measure, don't guess
Same prompt, same checkable-answer question from Session 10, sent to Groq and to your local model. Read your own numbers — they depend on your laptop, not on a published benchmark.

In [6]:
results = {}
try:
    results["groq"] = time_stream(groq, GROQ_MODEL)
except Exception as e:
    print("groq failed:", e)
if ollama and local_models:
    try:
        results["ollama"] = time_stream(ollama, local_models[0])
    except Exception as e:
        print("ollama failed:", e)
else:
    print("ollama skipped (not running)")

if results:
    print(speed_table(results))

provider     TTFT s   total s   tokens    tok/s
groq           1.39      1.46      543    373.2
ollama         5.42     14.79       53      3.6


*Question 4 — Your laptop's tokens/second is probably 5-15x slower than Groq's. For which of the three jobs from Session 10 (FAQ bot, document reader, agent-assist) does that gap matter least, and why?*

I found that the latency difference matters least for a Document Reader because document processing can usually happen as a background task. For real-time applications such as FAQ bots or live agent assistance, faster responses are more important.